### Cell 1 - Initialize Ray endpoints and verify dashboard

Installs requests, derives the Ray head host from RAY_ADDRESS, builds Dashboard/Serve/MLflow URLs, reads an Hugging Face token, and prints the endpoints plus the Jobs API version for a quick health check.

In [ ]:
!pip -q install requests==2.* --disable-pip-version-check

import os, textwrap, base64, time, json, requests
from string import Template

raw_addr = os.getenv("RAY_ADDRESS", "ray://ai-starter-kit-kuberay-head-svc:10001")
if raw_addr.startswith("ray://"):
    HEAD_HOST = raw_addr.split("://", 1)[1].split(":", 1)[0]
else:
    HEAD_HOST = raw_addr.split(":", 1)[0] or "ai-starter-kit-kuberay-head-svc"

DASH_URL    = f"http://{HEAD_HOST}:8265"
SERVE_PORT  = int(os.getenv("SERVE_PORT", "8000"))
SERVE_ROUTE = "/v1"

HF_TOKEN_PATH = "/etc/secrets/huggingface/token"
HF_TOKEN = ""
if os.path.exists(HF_TOKEN_PATH):
    try:
        HF_TOKEN = open(HF_TOKEN_PATH).read().strip()
    except Exception:
        HF_TOKEN = ""

print("Head host:", HEAD_HOST)
print("Jobs API :", f"{DASH_URL}/api/jobs/")
print("Serve URL:", f"http://{HEAD_HOST}:{SERVE_PORT}{SERVE_ROUTE}")
print("MLflow   :", os.getenv("MLFLOW_TRACKING_URI", "http://ai-starter-kit-mlflow:5000"))

print("Jobs API version:", requests.get(f"{DASH_URL}/api/version", timeout=10).json())


### Cell 2 - Deploy a minimal Ray Serve smoke test and verify readiness

Submits a tiny FastAPI app to Ray Serve (one /healthz endpoint under /smoke) as a Ray Job, installing FastAPI on the fly. It polls the Jobs API for status and hits :8000/smoke/healthz up to 60 seconds, printing when the service responds 200 (i.e., smoke test passes).

In [ ]:
import os, base64, textwrap, time, requests

DASH_URL = "http://ai-starter-kit-kuberay-head-svc:8265"

print("Jobs API:", requests.get(f"{DASH_URL}/api/version", timeout=10).json())

serve_py = textwrap.dedent("""
    from fastapi import FastAPI
    from ray import serve
    serve.start(detached=True, http_options={"host":"0.0.0.0","port":8000})
    app = FastAPI()

    @serve.deployment(name="smoke", num_replicas=1)
    @serve.ingress(app)
    class Smoke:
        @app.get("/healthz")
        async def health(self): return {"ok": True}

    serve.run(Smoke.bind(), route_prefix="/smoke")
    print("READY: smoke", flush=True)
""").strip()

b64 = base64.b64encode(serve_py.encode()).decode()
entry = f'python -c "import base64; exec(base64.b64decode(\'{b64}\'))"'
submit = requests.post(f"{DASH_URL}/api/jobs/", json={"entrypoint": entry, "runtime_env": {"pip": ["fastapi>=0.110"]}}, timeout=60).json()
job_id = submit["job_id"]
print("Job:", job_id)

svc = "http://ai-starter-kit-kuberay-head-svc:8000/smoke/healthz"
for i in range(60):
    s = requests.get(f"{DASH_URL}/api/jobs/{job_id}", timeout=10).json()["status"]
    try:
        r = requests.get(svc, timeout=2)
        print(f"tick {i:02d}: job={s}, health={r.status_code}")
        if r.status_code == 200:
            print("Smoke OK")
            break
    except Exception as e:
        print(f"tick {i:02d}: job={s}, health=ERR {e}")
    time.sleep(1)

### Cell 3 - Deploy model on Ray Serve with llama-cpp

Packages and submits a Ray Job that spins up a Ray Serve app exposing /v1/healthz and /v1/chat/completions. It downloads the preferred GGUF from Hugging Face, initializes llama-cpp-python, logs to MLflow, and prints the deployed health/chat URLs.

In [ ]:
import os, base64, textwrap, requests, time

HEAD        = os.environ.get("RAY_HEAD_SVC", "ai-starter-kit-kuberay-head-svc")
DASH_URL    = f"http://{HEAD}:8265"
SERVE_PORT  = 8000
SERVE_ROUTE = "/v1"

runtime_env = {
    "pip": [
        "fastapi==0.110.0",
        "uvicorn==0.23.2",
        "transformers==4.57.1",
        "torch==2.4.0",
        "accelerate==0.33.0",
        "mlflow==2.14.3",
    ],
    "env_vars": {
        "HUGGINGFACE_HUB_TOKEN": os.environ.get("HUGGINGFACE_HUB_TOKEN", ""),
        "SERVE_PORT": str(SERVE_PORT),
        "MODEL_NAME": "Qwen/Qwen2.5-1.5B-Instruct",
        "LLM_MAX_TOKENS": os.environ.get("LLM_MAX_TOKENS", "256"),
        "SERVER_MAX_NEW_TOKENS": os.environ.get("SERVER_MAX_NEW_TOKENS", "512"),
        "HF_HOME": "/tmp/hf-cache",
        "TRANSFORMERS_CACHE": "/tmp/hf-cache",
        "MLFLOW_TRACKING_URI": os.environ.get("MLFLOW_TRACKING_URI", ""),
        "MLFLOW_EXPERIMENT_NAME": os.environ.get("MLFLOW_EXPERIMENT_NAME", "ray-transformers"),
    },
}

serve_py = textwrap.dedent(f"""
import os, time, uuid
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from transformers import AutoModelForCausalLM, AutoTokenizer
from ray import serve

USE_MLFLOW = False
try:
    import mlflow
    if os.getenv("MLFLOW_TRACKING_URI"):
        mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
        mlflow.set_experiment(os.getenv("MLFLOW_EXPERIMENT_NAME","ray-transformers"))
        USE_MLFLOW = True
except Exception:
    USE_MLFLOW = False

SERVE_PORT  = int(os.getenv("SERVE_PORT", "{SERVE_PORT}"))
SERVE_ROUTE = "{SERVE_ROUTE}"
MODEL_NAME  = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
MAX_TOKENS  = int(os.getenv("LLM_MAX_TOKENS", "256"))
SERVER_MAX  = int(os.getenv("SERVER_MAX_NEW_TOKENS", "512"))

serve.start(detached=True, http_options={{"host":"0.0.0.0", "port":SERVE_PORT}})
app = FastAPI()

@serve.deployment(name="qwen", num_replicas=1, ray_actor_options={{"num_cpus": 4}})
@serve.ingress(app)
class TransformersLLM:
    def __init__(self):
        print(f"[load] Loading model: {{MODEL_NAME}}", flush=True)
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="cpu",
            trust_remote_code=True,
            torch_dtype="auto"
        )
        self.model_name = MODEL_NAME
        print("[ready] Model loaded", flush=True)

    @app.get("/healthz")
    async def health(self):
        return {{"status":"ok", "model": self.model_name}}

    @app.post("/chat/completions")
    async def chat_completions(self, request: Request):
        t0 = time.time()
        body = await request.json()

        messages = body.get("messages", [])
        temperature = float(body.get("temperature", 0.7))
        max_tokens = min(body.get("max_tokens", MAX_TOKENS), SERVER_MAX)

        try:
            prompt = self.tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            inputs = self.tokenizer(prompt, return_tensors="pt")
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
            )
            
            response_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response_text = response_text[len(prompt):].strip()
            
            p_tokens = len(inputs.input_ids[0])
            c_tokens = len(outputs[0]) - p_tokens
            
            if USE_MLFLOW:
                try:
                    with mlflow.start_run(run_name="chat"):
                        mlflow.log_params({{"temperature": temperature, "max_tokens": max_tokens}})
                        mlflow.log_metrics({{
                            "duration_ms": int((time.time()-t0) * 1000),
                            "prompt_tokens": p_tokens,
                            "completion_tokens": c_tokens,
                            "total_tokens": p_tokens + c_tokens
                        }})
                except Exception:
                    pass
            
            return {{
                "id": "chatcmpl-" + uuid.uuid4().hex[:24],
                "object": "chat.completion",
                "created": int(time.time()),
                "model": self.model_name,
                "choices": [{{
                    "index": 0,
                    "message": {{"role":"assistant","content": response_text}},
                    "finish_reason": "stop"
                }}],
                "usage": {{
                    "prompt_tokens": p_tokens,
                    "completion_tokens": c_tokens,
                    "total_tokens": p_tokens + c_tokens
                }}
            }}
        except Exception as e:
            return JSONResponse(status_code=500, content={{"error": str(e)}})

serve.run(TransformersLLM.bind(), route_prefix=SERVE_ROUTE)
print("READY", flush=True)
""").strip()

payload = base64.b64encode(serve_py.encode()).decode()
entrypoint = 'python -c "import base64,sys;exec(base64.b64decode(\'{}\').decode())"'.format(payload)

print("Submitting Ray job...")
job = requests.post(
    f"{DASH_URL}/api/jobs/",
    json={
        "entrypoint": entrypoint,
        "runtime_env": runtime_env,
        "metadata": {"job_name": "serve-qwen2_5-transformers"},
    },
    timeout=45
).json()

job_id = job.get("job_id")
print(f"Job ID: {job_id}")
print(f"Dashboard: {DASH_URL}/#/jobs/{job_id}")

print("\nWaiting for job to complete...")
print("\nThis may take several minutes.")
for i in range(300):
    time.sleep(2)
    
    try:
        status_resp = requests.get(f"{DASH_URL}/api/jobs/{job_id}", timeout=10)
        job_info = status_resp.json()
        
        status = job_info.get("status")
        message = job_info.get("message", "")
        
        if i % 15 == 0:
            print(f"  [{i*2}s] Status: {status}")
        
        if status == "SUCCEEDED":
            print(f"\nJob completed successfully after {i*2} seconds")
            print(f"Health: http://{HEAD}:{SERVE_PORT}{SERVE_ROUTE}/healthz")
            print(f"Chat:   http://{HEAD}:{SERVE_PORT}{SERVE_ROUTE}/chat/completions")
            break
        elif status == "FAILED":
            print(f"\nJob FAILED after {i*2} seconds")
            print(f"Error message: {message}")
            
            logs_resp = requests.get(f"{DASH_URL}/api/jobs/{job_id}/logs", timeout=10)
            if logs_resp.status_code == 200:
                logs = logs_resp.text
                print("\nLast 50 lines of logs:")
                print("=" * 60)
                print("\n".join(logs.split("\n")[-50:]))
            
            raise Exception(f"Ray job failed: {message}")
        elif status in ["STOPPED", "PENDING"]:
            if i > 150:
                print(f"\nJob stuck in {status} state for too long")
                raise Exception(f"Job did not start properly: {status}")
                
    except requests.exceptions.RequestException as e:
        print(f"  Error checking job status: {e}")
        continue
else:
    print(f"\nJob did not complete within 600 seconds")
    print(f"Current status: {status}")
    print(f"Check dashboard: {DASH_URL}/#/jobs/{job_id}")
    raise Exception("Job timeout")

### Cell 4 - Basic client + latency test

Calls /v1/healthz and then sends an OpenAI-style chat request to /v1/chat/completions with a short prompt. Prints latency and token usage, returning the assistant text.

In [ ]:
import os, time, requests, json

HEAD       = os.environ.get("RAY_HEAD_SVC", "ai-starter-kit-kuberay-head-svc")
SERVE_PORT = 8000
BASE_URL   = f"http://{HEAD}:{SERVE_PORT}/v1"

def health():
    r = requests.get(f"{BASE_URL}/healthz", timeout=10)
    print("Health:", r.status_code, r.json())

def chat(prompt, temperature=0.4, max_tokens=220, stop=None):
    body = {
        "model": "qwen2.5-1.5b-instruct-gguf",
        "temperature": float(temperature),
        "max_tokens": int(max_tokens),
        "messages": [
            {"role": "system", "content": "You are Qwen2.5 Instruct running on a tiny CPU host. Be concise, complete sentences."},
            {"role": "user", "content": prompt},
        ],
    }
    if stop:
        body["stop"] = stop

    t0 = time.time()
    r = requests.post(f"{BASE_URL}/chat/completions", json=body, timeout=300)
    dt = time.time() - t0
    r.raise_for_status()
    out = r.json()["choices"][0]["message"]["content"]
    usage = r.json().get("usage", {})
    print(f"\nLatency: {dt:.2f}s  | usage: {usage}")
    print("\n---\n", out)
    return out

health()
_ = chat("Say 'test ok' then give me one short fun fact about llamas.", stop=["<|im_end|>"])

### Cell 5 - Multi-agent (Autogen) pipeline

Installs Autogen, configures OpenAIWrapper to hit Ray Serve /v1 endpoint, warms up the model, then runs a simple three-agent workflow (Researcher -> Writer -> Critic) to produce and refine a short report.

In [ ]:
import os, requests, json, time

HEAD = os.environ.get("RAY_HEAD_SVC", "ai-starter-kit-kuberay-head-svc")
SERVE_PORT = 8000
BASE_URL = f"http://{HEAD}:{SERVE_PORT}/v1"

def call_llm(role_prompt, user_message, temperature=0.4, max_tokens=150):
    body = {
        "model": "qwen2.5-1.5b-instruct-gguf",
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": role_prompt},
            {"role": "user", "content": user_message}
        ]
    }
    try:
        r = requests.post(f"{BASE_URL}/chat/completions", json=body, timeout=120)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"Error: {e}"

# Try to use autogen if available, otherwise use direct implementation
USE_AUTOGEN = False

try:
    import autogen
    from autogen import AssistantAgent, UserProxyAgent
    USE_AUTOGEN = True
    print("Using autogen for multi-agent workflow")
except ImportError:
    try:
        print("Installing autogen dependencies...")
        !pip install -q pyautogen~=0.2.35 python-dotenv tiktoken "numpy<2,>=1.17.0" --disable-pip-version-check 2>/dev/null
        import autogen
        from autogen import AssistantAgent, UserProxyAgent
        USE_AUTOGEN = True
        print("Autogen installed successfully")
    except:
        print("Using direct implementation (autogen not available)")
        USE_AUTOGEN = False

if USE_AUTOGEN:
    config_list = [
        {
            "model": "qwen2.5-1.5b-instruct-gguf",
            "base_url": BASE_URL,
            "api_key": "local",
            "price": [0.0, 0.0],
        }
    ]
    
    llm = autogen.OpenAIWrapper(config_list=config_list)
    
    try:
        r = llm.create(messages=[{"role":"user","content":"Say 'test ok'."}], temperature=0.2, max_tokens=16)
        print("Warmup:", r.choices[0].message.content)
    except Exception as e:
        print("Warmup skipped:", e)
    
    user_proxy = UserProxyAgent(
        name="UserProxy",
        system_message="You are the human admin. Initiate the task.",
        code_execution_config=False,
        human_input_mode="NEVER",
    )
    
    researcher = AssistantAgent(
        name="Researcher",
        system_message=(
            "You are a researcher. Gather concise, verified facts on the topic. "
            "Return 3-4 bullet points. Keep under 100 words total."
        ),
        llm_config={"config_list": config_list, "temperature": 0.35, "max_tokens": 140, "timeout": 120},
    )
    
    writer = AssistantAgent(
        name="Writer",
        system_message=(
            "You are a writer. Using the Researcher's notes, produce a clear report under 160 words."
        ),
        llm_config={"config_list": config_list, "temperature": 0.55, "max_tokens": 220, "timeout": 180},
    )
    
    critic = AssistantAgent(
        name="Critic",
        system_message=(
            "You are a critic. Review the Writer's report for accuracy and clarity. "
            "Present the final polished text under 140 words."
        ),
        llm_config={"config_list": config_list, "temperature": 0.45, "max_tokens": 160, "timeout": 120},
    )
    
    def run_sequential(task):
        print("\n" + "=" * 60)
        print("Running Multi-Agent Workflow (with autogen)")
        print("=" * 60)
        
        research_response = researcher.generate_reply(messages=[{"content": task, "role": "user"}])
        research_notes = research_response if isinstance(research_response, str) else research_response.get("content", "[no output]")
        print("\n1. RESEARCHER:")
        print("-" * 40)
        print(research_notes)
        
        writer_prompt = f"Using these research notes, write the report:\n{research_notes}"
        writer_response = writer.generate_reply(messages=[{"content": writer_prompt, "role": "user"}])
        report = writer_response if isinstance(writer_response, str) else writer_response.get("content", "[no output]")
        print("\n2. WRITER:")
        print("-" * 40)
        print(report)
        
        critic_prompt = f"Review this report:\n{report}"
        critic_response = critic.generate_reply(messages=[{"content": critic_prompt, "role": "user"}])
        final_text = critic_response if isinstance(critic_response, str) else critic_response.get("content", "[no output]")
        print("\n3. CRITIC/EDITOR:")
        print("-" * 40)
        print(final_text)
        return final_text
    
    task = "Research the latest advancements in quantum computing as of 2025. Gather key facts, then write a short report."
    final_output = run_sequential(task)
    
else:
    print("=" * 60)
    print("Running Multi-Agent Workflow (direct implementation)")
    print("=" * 60)
    
    task = "Research the latest advancements in quantum computing as of 2025."
    
    print("\n1. RESEARCHER:")
    print("-" * 40)
    research_prompt = "You are a researcher. Provide 3-4 key facts about the topic. Be concise and factual."
    research_notes = call_llm(research_prompt, task, temperature=0.35, max_tokens=140)
    print(research_notes)
    time.sleep(1) 
    
    print("\n2. WRITER:")
    print("-" * 40)
    writer_prompt = "You are a technical writer. Based on the following notes, write a brief report."
    writer_task = f"Write a report based on these notes:\n{research_notes}"
    report = call_llm(writer_prompt, writer_task, temperature=0.55, max_tokens=220)
    print(report)
    time.sleep(1)
    
    print("\n3. CRITIC/EDITOR:")
    print("-" * 40)
    critic_prompt = "You are an editor. Review the report and provide a final polished version."
    critic_task = f"Review and improve this report:\n{report}"
    final_output = call_llm(critic_prompt, critic_task, temperature=0.45, max_tokens=160)
    print(final_output)

print("\n" + "=" * 60)
print("Multi-agent workflow complete")
print("=" * 60)

### Cell 6 - MLFlow: connect to tracking server and list recent chat runs

Installs MLflow, sets the tracking URI and experiment, then queries and prints the latest runs with key params/metrics (temperature, max_tokens, duration) to verify Serve logging.

In [ ]:
!pip -q install mlflow==2.14.3 --disable-pip-version-check

import os, mlflow
from datetime import datetime

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://ai-starter-kit-mlflow:5000")
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow Tracking URI: {tracking_uri}")

exp_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "ray-transformers")
exp = mlflow.set_experiment(exp_name)
print(f"Experiment: {exp.name} (ID: {exp.experiment_id})")
print("-" * 60)

client = mlflow.tracking.MlflowClient()
runs = client.search_runs(
    exp.experiment_id, 
    order_by=["attributes.start_time DESC"], 
    max_results=10
)

if not runs:
    print("No runs found. Run cells 4 or 5 first to generate inference requests.")
else:
    print(f"\nFound {len(runs)} recent runs:")
    print("-" * 60)
    
    for i, run in enumerate(runs, 1):
        start_time = datetime.fromtimestamp(run.info.start_time/1000).strftime('%Y-%m-%d %H:%M:%S')
        duration = run.data.metrics.get('duration_ms', 'N/A')
        temp = run.data.params.get('temperature', 'N/A')
        max_tokens = run.data.params.get('max_tokens', 'N/A')
        total_tokens = run.data.metrics.get('total_tokens_approx', 'N/A')
        
        print(f"\nRun {i}:")
        print(f"  ID:          {run.info.run_id[:12]}...")
        print(f"  Time:        {start_time}")
        print(f"  Status:      {run.info.status}")
        print(f"  Temperature: {temp}")
        print(f"  Max Tokens:  {max_tokens}")
        print(f"  Duration:    {duration} ms")
        print(f"  Total Tokens: {total_tokens}")
    
    print("\n" + "=" * 60)
    print("SUMMARY:")
    successful = sum(1 for r in runs if r.info.status == 'FINISHED')
    durations = [r.data.metrics.get('duration_ms', 0) for r in runs if r.data.metrics.get('duration_ms')]
    avg_duration = sum(durations) / len(durations) if durations else 0
    
    print(f"  Total Runs: {len(runs)}")
    print(f"  Successful: {successful}")
    print(f"  Failed: {len(runs) - successful}")
    print(f"  Avg Duration: {avg_duration:.1f} ms" if avg_duration else "  Avg Duration: N/A")

print("\n" + "=" * 60)
print("MLflow verification complete")